# Étude comparative de l'EM classique et de l'EM variationnel
Kana Tiodong Icare Brichel, Master 2 SDIA, ENSPD, Université de Douala

**Objet du carnet.** Produire l'ensemble des résultats de l'étude avec des implémentations écrites d'après les équations du texte (`algorithmes.py`, PyTorch seul), les présenter avec un protocole explicite et des incertitudes chiffrées, et générer les tableaux et figures du document final.

Exécution : *Run All*. Durée : environ 4 minutes (réglage `RAPIDE = True` pour un essai en 1 minute). Les sorties sont écrites dans `resultats/` ; les figures du document sont exportées dans `figures/`.

## 1. Protocole

**Question de recherche.** Quand le nombre de composantes d'un mélange gaussien est surestimé, l'EM variationnel apporte-t-il un gain mesurable par rapport à l'EM classique, et ce gain dépend-il des hyperparamètres des priors ?

**Hypothèses** (fixées avant l'analyse).

| Code | Hypothèse | Critère de décision |
|---|---|---|
| H1 | À K = 6 imposé, l'ARI moyen de l'EM variationnel dépasse celui de l'EM classique | Intervalle de confiance à 95 % (bootstrap) de la différence appariée strictement positif |
| H2 | L'EM variationnel retrouve exactement 3 composantes actives dans la majorité des jeux | Borne basse de l'intervalle de Wilson à 95 % supérieure à 0,5 |
| H3 | Les résultats de H1 et H2 dépendent des priors (diffus : beta0 = 1e-3, W0 = I ; empiriques : beta0 = 1, W0 = covariance empirique) | Comparaison des deux jeux de priors sur les mêmes données |

**Données.** Mélange de trois gaussiennes isotropes, centres (0, 0), (5, 5), (0, 5), écart-type 0,7, 100 observations par groupe (N = 300). Jeu de référence : graine 42. Réplication : graines 1000 à 1029. Jeu réel : Iris (N = 150, D = 4).

**Algorithmes.** EM classique (équations 1.15 et 1.16), EM variationnel (1.26 à 1.30), ELBO (1.31). Composante *active* : N_k = somme des responsabilités >= 1.

**Réglages communs.** 10 initialisations par ajustement, meilleur critère retenu (log-vraisemblance ou ELBO) ; arrêt quand la variation du critère passe sous 1e-3 (par observation pour l'EM, valeur totale pour l'EM variationnel) ; au plus 200 à 500 itérations ; alpha0 = 1/K.

**Métriques.** Indice de Rand ajusté (ARI), silhouette, nombre de composantes actives, itérations, temps. Incertitudes : bootstrap (5 000 rééchantillonnages) pour les moyennes, intervalle de Wilson pour les proportions.

**Limites assumées.** Un seul schéma génératif ; dimensions modestes ; temps mesurés sur une machine CPU unique (comparaisons relatives seulement).

In [2]:
import sys, pathlib, shutil
sys.path.insert(0, str(pathlib.Path.cwd()))
import pandas as pd, torch
from IPython.display import display, Markdown
import algorithmes as A
import main as M

RAPIDE = False          # True : tailles et répétitions réduites

def figure(nom):
    """Affiche la figure Plotly interactive produite par l'expérience."""
    M.FIGURES[nom].show()

def tableau(nom):
    display(pd.read_csv(M.SORTIE / f"{nom}.csv"))

## 2. Environnement et reproductibilité

In [3]:
env = M.environnement()
display(pd.Series(env, name="valeur").to_frame())
print("Graines : données 42 (référence) et 1000 à 1029 (réplication) ; initialisations : graine + indice de relance.")

,valeur
python,3.10.0
torch,2.4.1+cpu
pandas,2.1.4
plotly,6.8.0
systeme,Windows-10-10.0.21996-SP0
cpu_threads,4
precision,torch.float64


Graines : données 42 (référence) et 1000 à 1029 (réplication) ; initialisations : graine + indice de relance.


## 3. Validation des implémentations
Avant toute comparaison, on vérifie que le code respecte les propriétés théoriques du chapitre 1 : croissance de la log-vraisemblance (EM) et de l'ELBO (EM variationnel), égalité F(q*, theta) = log-vraisemblance après l'étape E (Neal et Hinton), tableaux 1.4 (forward-backward) et 1.5, exemple à 8 observations.

In [4]:
M.exp00_validation()
figure("00_log_vraisemblance_8_points")
tableau("00_tableau_1_2_em_8_points")


00  Validation des exemples numériques du chapitre 1
Tableau 1.2 recalculé (EM classique, 8 observations) :
 iteration   mu1   mu2  sigma1  sigma2   pi1   pi2  loglik
         0 2.000 6.000   2.000   2.000 0.500 0.500     NaN
         1 1.284 7.368   1.134   1.549 0.484 0.516 -15.814
         2 1.125 7.722   0.466   0.479 0.500 0.500 -10.742
         3 1.125 7.725   0.466   0.460 0.500 0.500 -10.736
         4 1.125 7.725   0.466   0.460 0.500 0.500 -10.736
         5 1.125 7.725   0.466   0.460 0.500 0.500 -10.736
         6 1.125 7.725   0.466   0.460 0.500 0.500 -10.736
         7 1.125 7.725   0.466   0.460 0.500 0.500 -10.736
         8 1.125 7.725   0.466   0.460 0.500 0.500 -10.736
         9 1.125 7.725   0.466   0.460 0.500 0.500 -10.736
        10 1.125 7.725   0.466   0.460 0.500 0.500 -10.736
Moyennes empiriques des deux groupes : 1.125 et 7.725
Densité du mélange ajusté en x = 2,0 (éq. 1.10) : 0.0734
Dirichlet a posteriori [8.0, 5.0, 2.0] ; E[pi] = [0.533, 0.333, 0.133]
T

,iteration,mu1,mu2,sigma1,sigma2,pi1,pi2,loglik
0,0,2.000,6.000,2.000,2.000,0.500,0.500,NaN
1,1,1.284,7.368,1.134,1.549,0.484,0.516,-15.814
2,2,1.125,7.722,0.466,0.479,0.500,0.500,-10.742
3,3,1.125,7.725,0.466,0.460,0.500,0.500,-10.736
4,4,1.125,7.725,0.466,0.460,0.500,0.500,-10.736
5,5,1.125,7.725,0.466,0.460,0.500,0.500,-10.736
6,6,1.125,7.725,0.466,0.460,0.500,0.500,-10.736
7,7,1.125,7.725,0.466,0.460,0.500,0.500,-10.736
8,8,1.125,7.725,0.466,0.460,0.500,0.500,-10.736
9,9,1.125,7.725,0.466,0.460,0.500,0.500,-10.736


## 4. Résultats sur le jeu de référence (K = 6 imposé)
**Lecture.** Chaque couleur est une composante ; l'ellipse est à deux écarts-types ; la croix marque le centre. Une composante *élaguée* a N_k < 1 et n'apparaît pas sur le nuage.

In [5]:
M.exp01_comparaison_k6()
figure("01_jeu_donnees")
figure("01_ellipses")
figure("01_poids")
tableau("01_comparaison_k6")


01  Comparaison K = 6 (sections 2.3.1 à 2.3.4)
                                         modele  composantes_actives   ARI  silhouette  iterations  converge  degenere  temps_s                                      poids
       EM classique (init k-means, 10 relances)                    6 0.794       0.390         162      True     False    2.568 [0.258, 0.332, 0.083, 0.013, 0.238, 0.075]
   EM classique (init aléatoire, version 3.2.1)                    6 0.795       0.407         119      True     False    0.218  [0.333, 0.08, 0.303, 0.074, 0.034, 0.176]
  EM variationnel (priors diffus, init k-means)                    4 0.985       0.646         102      True      True    3.139 [0.001, 0.326, 0.333, 0.332, 0.008, 0.001]
            EM variationnel (priors empiriques)                    3 0.990       0.739          22      True     False    1.015 [0.001, 0.331, 0.333, 0.335, 0.001, 0.001]
EM variationnel (init aléatoire, version 3.3.1)                    4 0.873       0.590         12

,modele,composantes_actives,ARI,silhouette,iterations,converge,degenere,temps_s,poids
0,"EM classique (init k-means, 10 relances)",6,0.794,0.390,162,True,False,2.568,"[0.258, 0.332, 0.083, 0.013, 0.238, 0.075]"
1,"EM classique (init aléatoire, version 3.2.1)",6,0.795,0.407,119,True,False,0.218,"[0.333, 0.08, 0.303, 0.074, 0.034, 0.176]"
2,"EM variationnel (priors diffus, init k-means)",4,0.985,0.646,102,True,True,3.139,"[0.001, 0.326, 0.333, 0.332, 0.008, 0.001]"
3,EM variationnel (priors empiriques),3,0.990,0.739,22,True,False,1.015,"[0.001, 0.331, 0.333, 0.335, 0.001, 0.001]"
4,"EM variationnel (init aléatoire, version 3.3.1)",4,0.873,0.590,129,True,False,0.447,"[0.198, 0.001, 0.001, 0.332, 0.334, 0.135]"


## 5. Sensibilité au prior de Dirichlet alpha0
**Lecture.** L'axe horizontal est logarithmique. Chaque courbe correspond à un jeu de priors ; le nombre vrai de groupes est 3.

In [6]:
M.exp02_alpha0()
figure("02_alpha0")
tableau("02_sensibilite_alpha0")


02  Sensibilité au prior de Dirichlet alpha0 (section 2.3.5)
 alpha0            priors  composantes_actives   ARI                                      poids  iterations
  0.001     priors diffus                    3 1.000       [0.0, 0.335, 0.333, 0.332, 0.0, 0.0]         100
  0.001 priors empiriques                    3 0.990       [0.0, 0.335, 0.333, 0.331, 0.0, 0.0]          23
  0.010     priors diffus                    3 1.000       [0.0, 0.334, 0.333, 0.332, 0.0, 0.0]         100
  0.010 priors empiriques                    3 0.990       [0.0, 0.335, 0.331, 0.333, 0.0, 0.0]          24
  0.100     priors diffus                    3 1.000       [0.0, 0.334, 0.333, 0.332, 0.0, 0.0]         101
  0.100 priors empiriques                    3 0.990       [0.0, 0.335, 0.333, 0.331, 0.0, 0.0]          23
  1.000     priors diffus                    4 0.981   [0.014, 0.329, 0.33, 0.32, 0.003, 0.003]         101
  1.000 priors empiriques                    3 0.990  [0.33, 0.003, 0.328,

,alpha0,priors,composantes_actives,ARI,poids,iterations
0,0.001,priors diffus,3,1.000,"[0.0, 0.335, 0.333, 0.332, 0.0, 0.0]",100
1,0.001,priors empiriques,3,0.990,"[0.0, 0.335, 0.333, 0.331, 0.0, 0.0]",23
2,0.010,priors diffus,3,1.000,"[0.0, 0.334, 0.333, 0.332, 0.0, 0.0]",100
3,0.010,priors empiriques,3,0.990,"[0.0, 0.335, 0.331, 0.333, 0.0, 0.0]",24
4,0.100,priors diffus,3,1.000,"[0.0, 0.334, 0.333, 0.332, 0.0, 0.0]",101
5,0.100,priors empiriques,3,0.990,"[0.0, 0.335, 0.333, 0.331, 0.0, 0.0]",23
6,1.000,priors diffus,4,0.981,"[0.014, 0.329, 0.33, 0.32, 0.003, 0.003]",101
7,1.000,priors empiriques,3,0.990,"[0.33, 0.003, 0.328, 0.332, 0.003, 0.003]",23
8,10.000,priors diffus,6,0.618,"[0.185, 0.162, 0.167, 0.166, 0.171, 0.149]",55
9,10.000,priors empiriques,6,0.960,"[0.075, 0.263, 0.211, 0.306, 0.074, 0.072]",168


## 6. Scalabilité
Les temps mélangent coût par itération et nombre d'itérations ; ils servent à comparer les deux algorithmes entre eux, pas à prédire un temps absolu.

In [7]:
M.exp03_scalabilite(RAPIDE)
figure("03_scalabilite")
tableau("03_scalabilite")


03  Scalabilité (section 2.3.6)
 parametre  valeur  n_rep  classique_s  variationnel_s  ratio  iter_classique  iter_variationnel  cout_iter_classique_s  cout_iter_variationnel_s  ratio_cout_iter
   N (D=2)     300     10       0.2306          0.2041    0.9           154.0               58.0                0.00150                   0.00352              2.4
   N (D=2)    1000     10       0.3181          0.5574    1.8           155.0              139.0                0.00205                   0.00401              2.0
   N (D=2)    5000     10       0.3474          1.9507    5.6            54.0              299.0                0.00643                   0.00652              1.0
   N (D=2)   20000     10       0.6908          1.1770    1.7            56.0               88.0                0.01234                   0.01338              1.1
D (N=2000)       2     10       0.3349          1.1333    3.4            98.0              226.0                0.00342                   0.00501       

,parametre,valeur,n_rep,classique_s,variationnel_s,ratio,iter_classique,iter_variationnel,cout_iter_classique_s,cout_iter_variationnel_s,ratio_cout_iter
0,N (D=2),300,10,0.2306,0.2041,0.9,154.0,58.0,0.00150,0.00352,2.4
1,N (D=2),1000,10,0.3181,0.5574,1.8,155.0,139.0,0.00205,0.00401,2.0
2,N (D=2),5000,10,0.3474,1.9507,5.6,54.0,299.0,0.00643,0.00652,1.0
3,N (D=2),20000,10,0.6908,1.1770,1.7,56.0,88.0,0.01234,0.01338,1.1
4,D (N=2000),2,10,0.3349,1.1333,3.4,98.0,226.0,0.00342,0.00501,1.5
5,D (N=2000),5,10,0.4245,0.9600,2.3,127.0,199.0,0.00334,0.00482,1.4
6,D (N=2000),10,10,0.4424,0.3777,0.9,131.0,77.0,0.00338,0.00490,1.5
7,D (N=2000),20,10,0.3383,0.3660,1.1,98.0,62.0,0.00345,0.00590,1.7


## 7. Cas où K est correctement spécifié

In [8]:
M.exp04_k3()
tableau("04_k3")


04  Cas de référence K = 3 (section 2.3.7)
                modele  composantes_actives  ARI  silhouette  iterations  converge  degenere  temps_s                 poids
      EM classique K=3                    3 1.00       0.740          72      True     False    0.323 [0.332, 0.333, 0.335]
EM variationnel Kmax=3                    3 0.99       0.739           3      True     False    0.268 [0.333, 0.335, 0.331]


,modele,composantes_actives,ARI,silhouette,iterations,converge,degenere,temps_s,poids
0,EM classique K=3,3,1.00,0.740,72,True,False,0.323,"[0.332, 0.333, 0.335]"
1,EM variationnel Kmax=3,3,0.99,0.739,3,True,False,0.268,"[0.333, 0.335, 0.331]"


## 8. Convergence de l'ELBO
L'ELBO est calculée exactement (équation 1.31). Elle doit croître à chaque itération et rester inférieure à la log-vraisemblance maximale.

In [9]:
M.exp05_elbo()
figure("05_convergence")


05  Convergence de l'ELBO (section 2.3.8)
ELBO : première valeur -1134.71, dernière -1046.62, croissance stricte : True, itérations : 22


## 9. Robustesse à l'initialisation
Une seule initialisation par exécution (n_init = 1), graines 0 à 19.

In [10]:
M.exp06_initialisation(RAPIDE)
figure("06_initialisation")
tableau("06_initialisation")


06  Robustesse à l'initialisation (section 2.3.9)
[init k-means] log-vraisemblance EM : min -947.67, max -942.12, écart-type 1.61 ; ARI moyen EM variationnel 0.926
[init aléatoire] log-vraisemblance EM : min -948.33, max -935.57, écart-type 3.52 ; ARI moyen EM variationnel 0.921
initialisation                          algorithme  n_repetitions  3 actives  4 actives  5 actives  6 actives
  init k-means                        EM classique             20          0          0          0         20
  init k-means     EM variationnel (priors diffus)             20          0         12          8          0
  init k-means EM variationnel (priors empiriques)             20         20          0          0          0
init aléatoire                        EM classique             20          0          0          0         20
init aléatoire     EM variationnel (priors diffus)             20          4         14          2          0
init aléatoire EM variationnel (priors empiriques)         

,initialisation,algorithme,n_repetitions,3 actives,4 actives,5 actives,6 actives
0,init k-means,EM classique,20,0,0,0,20
1,init k-means,EM variationnel (priors diffus),20,0,12,8,0
2,init k-means,EM variationnel (priors empiriques),20,20,0,0,0
3,init aléatoire,EM classique,20,0,0,0,20
4,init aléatoire,EM variationnel (priors diffus),20,4,14,2,0
5,init aléatoire,EM variationnel (priors empiriques),20,20,0,0,0


## 10. Type de covariance

In [11]:
M.exp07_covariance()
figure("07_covariance")
tableau("07_covariance")


07  Sensibilité au type de covariance (section 2.3.10)
covariance  ARI_classique  ARI_variationnel  actives_variationnel  BIC_classique
      full          0.794              0.99                     3         2083.9
      tied          0.745              1.00                     3         2022.7
      diag          0.860              0.99                     3         2048.2
 spherical          0.809              1.00                     3         2022.1
Remarque : les variantes tied, diag et spherical de l'EM variationnel ne sont pas dérivées ici ; elles utilisent un prior Wishart partagé (tied) ou Gamma (diag, spherical), conjugués.


,covariance,ARI_classique,ARI_variationnel,actives_variationnel,BIC_classique
0,full,0.794,0.99,3,2083.9
1,tied,0.745,1.00,3,2022.7
2,diag,0.860,0.99,3,2048.2
3,spherical,0.809,1.00,3,2022.1


## 11. Jeu de données réel : Iris

In [12]:
M.exp08_iris()
figure("08_iris")
tableau("08_iris")


08  Jeu de données réel Iris (section 2.3.11)
                modele  composantes_actives   ARI  silhouette  iterations  converge  degenere  temps_s                                      poids
      EM classique K=6                    6 0.662       0.440          34      True      True    1.121  [0.333, 0.226, 0.02, 0.053, 0.191, 0.177]
EM variationnel Kmax=6                    3 0.562       0.276         137      True     False    2.008 [0.007, 0.073, 0.001, 0.001, 0.585, 0.332]
      EM classique K=3                    3 0.904       0.501          22      True     False    0.265                      [0.333, 0.367, 0.299]
EM variationnel Kmax=3                    3 0.644       0.368         104      True     False    3.111                       [0.16, 0.333, 0.506]


,modele,composantes_actives,ARI,silhouette,iterations,converge,degenere,temps_s,poids
0,EM classique K=6,6,0.662,0.440,34,True,True,1.121,"[0.333, 0.226, 0.02, 0.053, 0.191, 0.177]"
1,EM variationnel Kmax=6,3,0.562,0.276,137,True,False,2.008,"[0.007, 0.073, 0.001, 0.001, 0.585, 0.332]"
2,EM classique K=3,3,0.904,0.501,22,True,False,0.265,"[0.333, 0.367, 0.299]"
3,EM variationnel Kmax=3,3,0.644,0.368,104,True,False,3.111,"[0.16, 0.333, 0.506]"


## 12. Valeurs aberrantes
Dix points tirés uniformément dans [-10 ; 15] x [-10 ; 15], graine 7.

In [13]:
M.exp09_aberrantes()
tableau("09_aberrantes")


09  Valeurs aberrantes (section 2.3.12)
         modele  ARI_hors_aberrantes  composantes_actives  trace_covariance_max  poids_plus_petit
   EM classique                 0.99                    6                   6.2            0.0065
EM variationnel                 0.99                    5                  98.5            0.0108


,modele,ARI_hors_aberrantes,composantes_actives,trace_covariance_max,poids_plus_petit
0,EM classique,0.99,6,6.2,0.0065
1,EM variationnel,0.99,5,98.5,0.0108


## 13. Sélection de modèle par BIC et AIC

In [14]:
M.exp10_bic_aic()
figure("10_bic_aic")
tableau("10_bic_aic")


10  Sélection de modèle par BIC et AIC (section 2.2.3)
 K   loglik     BIC     AIC
 1 -1348.44 2725.40 2706.88
 2 -1064.91 2192.56 2151.82
 3  -955.80 2008.56 1945.59
 4  -950.04 2031.27 1946.08
 5  -946.95 2059.30 1951.89
 6  -942.11 2083.85 1954.22
 7  -931.02 2095.89 1944.03
 8  -927.69 2123.46 1949.38
K* selon BIC : 3 ; K* selon AIC : 7


,K,loglik,BIC,AIC
0,1,-1348.44,2725.40,2706.88
1,2,-1064.91,2192.56,2151.82
2,3,-955.80,2008.56,1945.59
3,4,-950.04,2031.27,1946.08
4,5,-946.95,2059.30,1951.89
5,6,-942.11,2083.85,1954.22
6,7,-931.02,2095.89,1944.03
7,8,-927.69,2123.46,1949.38


## 14. Variantes de l'EM, Baum-Welch et LDA

In [15]:
M.exp11_variantes_hmm_lda(RAPIDE)
tableau("11_variantes")
tableau("11_lda_themes")
tableau("11_lda_documents")


11  Variantes de l'EM, Baum-Welch et LDA
                                               methode  loglik  ARI  temps_s
                                   EM standard (batch)  -955.8 1.00    0.064
EM stochastique (SEM), 20 baisses de log-vraisemblance  -955.8 1.00    0.300
                  EM online (statistiques suffisantes)  -956.0 1.00    0.070
             EM variationnel batch (Kmax=6, 3 actives)     NaN 0.99    0.261
                               SVI (Kmax=6, 3 actives)     NaN 0.99    0.266
Baum-Welch discret (T=1000) : log-vraisemblance croissante : True, 100 itérations
  log-vraisemblance estimée -662.74 contre -663.19 aux vrais paramètres (le maximum de vraisemblance peut dépasser la vraie valeur sur une séquence courte)
  A vrai [[0.7, 0.3], [0.4, 0.6]]  estimé [[0.705, 0.295], [0.383, 0.617]]
  B vrai [[0.1, 0.9], [0.8, 0.2]]  estimé [[0.105, 0.895], [0.804, 0.196]]
Baum-Welch gaussien : sigma estimés [0.482, 1.99] (vrais 0,5 et 2,0), A estimée [[0.939, 0.061], [0.164, 0.8

,methode,loglik,ARI,temps_s
0,EM standard (batch),-955.8,1.00,0.064
1,"EM stochastique (SEM), 20 baisses de log-vrais...",-955.8,1.00,0.300
2,EM online (statistiques suffisantes),-956.0,1.00,0.070
3,"EM variationnel batch (Kmax=6, 3 actives)",NaN,0.99,0.261
4,"SVI (Kmax=6, 3 actives)",NaN,0.99,0.266


,theme,mots
0,0,"football, reprend, end, remporte, week, equipe"
1,1,"taux, but, joueur, rencontre, decisif, marque"
2,2,"chat, chien, actions, progressent, marche, fin..."


,document,T0,T1,T2,theme_dominant,theme_vrai
0,le chat mange des croquettes et dort sur le ca...,0.96,0.02,0.02,0,0
1,le chien joue dans le jardin avec la balle,0.02,0.02,0.95,2,0
2,le chat et le chien dorment ensemble le soir,0.02,0.02,0.96,2,0
3,la bourse chute apres l annonce des taux d int...,0.02,0.02,0.96,2,1
4,les actions technologiques progressent sur le ...,0.02,0.02,0.96,2,1
5,la banque centrale releve son taux directeur c...,0.02,0.97,0.02,1,1
6,l equipe de football remporte le match a l ext...,0.96,0.02,0.02,0,2
7,le joueur marque un but decisif en fin de renc...,0.02,0.97,0.02,1,2
8,le championnat de football reprend ce week end,0.96,0.02,0.02,0,2
9,le chat observe les oiseaux depuis la fenetre ...,0.02,0.02,0.96,2,0


## 15. Réplication statistique
Les résultats des sections 4 à 13 portent sur un seul jeu de données. Cette section répète le protocole sur des jeux indépendants et teste H1 à H3.

In [16]:
M.exp12_replication(RAPIDE)
figure("12_replication")
tableau("12_replication_resume")
tableau("12_replication_differences")


12  Réplication statistique sur jeux de données indépendants
                             modele  R  ARI_moyen        ARI_IC95  ARI_ecart_type  silhouette_moyenne silhouette_IC95  part_exactement_3_actives part_IC95_Wilson  actives_moyennes  iterations_moyennes  temps_moyen_s
                       EM classique 30      0.760 [0.743 ; 0.777]           0.048               0.412 [0.389 ; 0.436]                      0.000  [0.000 ; 0.114]              6.00                128.5          2.277
    EM variationnel (priors diffus) 30      0.971 [0.953 ; 0.988]           0.050               0.709 [0.685 ; 0.732]                      0.733  [0.556 ; 0.858]              3.27                118.5          4.045
EM variationnel (priors empiriques) 30      0.999 [0.998 ; 1.000]           0.003               0.746 [0.743 ; 0.749]                      1.000  [0.886 ; 1.000]              3.00                 27.2          0.996
                                                 comparaison  difference_m

,modele,R,ARI_moyen,ARI_IC95,ARI_ecart_type,silhouette_moyenne,silhouette_IC95,part_exactement_3_actives,part_IC95_Wilson,actives_moyennes,iterations_moyennes,temps_moyen_s
0,EM classique,30,0.760,[0.743 ; 0.777],0.048,0.412,[0.389 ; 0.436],0.000,[0.000 ; 0.114],6.00,128.5,2.277
1,EM variationnel (priors diffus),30,0.971,[0.953 ; 0.988],0.050,0.709,[0.685 ; 0.732],0.733,[0.556 ; 0.858],3.27,118.5,4.045
2,EM variationnel (priors empiriques),30,0.999,[0.998 ; 1.000],0.003,0.746,[0.743 ; 0.749],1.000,[0.886 ; 1.000],3.00,27.2,0.996


,comparaison,difference_moyenne,IC95_bootstrap,jeux_ou_variationnel_meilleur
0,EM variationnel (priors diffus) moins EM class...,0.211,[0.192 ; 0.229],30 sur 30
1,EM variationnel (priors empiriques) moins EM c...,0.239,[0.222 ; 0.256],30 sur 30


## 16. Cas pratique : rendements du S&P 500 et risque de marché
**Question.** Un mélange gaussien décrit-il mieux les queues épaisses des rendements qu'une loi normale, et que gagne-t-on à utiliser la loi a posteriori de l'EM variationnel pour la value-at-risk ?

**Données.** Cours quotidiens de clôture de l'indice S&P 500 (source : Federal Reserve Bank of St. Louis, série FRED SP500, dix dernières années), rendement = 100 x ln(P_t / P_{t-1}). Apprentissage : jusqu'au 31 décembre 2021 (crise de 2020 comprise). Évaluation hors échantillon : depuis le 1er janvier 2022 (hausse des taux de 2022 et chocs récents).

**Hypothèses** (fixées avant l'analyse).

| Code | Hypothèse | Critère de décision |
|---|---|---|
| H4 | Un mélange (K > 1) a une log-vraisemblance hors échantillon supérieure à celle de la gaussienne unique | Écart de log-vraisemblance moyenne par observation > 0 |
| H5 | La VaR gaussienne à 1 % est franchie plus souvent que prévu ; la VaR du mélange est mieux calibrée | Test de Kupiec, seuil de 5 % |
| H6 | L'EM variationnel fournit un intervalle de crédibilité de la VaR ; l'EM classique n'en donne un que par bootstrap | Comparaison des deux intervalles à 95 % |

**Méthode.** EM classique pour K = 1 à 6 (K choisi par BIC), EM variationnel avec Kmax = 6 et priors empiriques, log-vraisemblance prédictive a posteriori (mélange de lois de Student), VaR et Expected Shortfall d'un mélange gaussien, tirages a posteriori de (pi, mu, sigma), bootstrap non paramétrique de l'EM (B = 100), Baum-Welch à deux états gaussiens pour les régimes.

**Limites.** Série non stationnaire ; paramètres fixes sur la période d'évaluation (pas de réestimation glissante) ; un seul indice.

In [17]:
import importlib, cas_pratique as C
importlib.reload(C)      # recharge le module apres toute modification du fichier
C.main(["--rapide"] if RAPIDE else [])
display(pd.read_csv(M.SORTIE / "cas_pratique_descriptif.csv"))
M.FIGURES["cas_distribution"].show()
M.FIGURES["cas_selection"].show()
display(pd.read_csv(M.SORTIE / "cas_pratique_modeles.csv"))


Cas pratique : rendements du S&P 500 et risque de marché
                periode  observations  moyenne_pct  ecart_type_pct  asymetrie  kurtosis_excedentaire  minimum_pct  maximum_pct
apprentissage 2016-2021          1318        0.060          1.1960     -1.154                 22.720       -12.77         8.97
   évaluation 2022-2026          1193        0.041          1.0786      0.039                  6.487        -6.16         9.09
                                                        modele  composantes  loglik_moy_apprentissage  loglik_moy_evaluation    BIC
                                 Gaussienne unique (EM, K = 1)            1                   -1.5976                -1.5043 4225.5
                        EM classique, K choisi par BIC (K = 4)            4                   -1.2876                -1.4620 3473.1
                                    EM classique, K = 6 imposé            6                   -1.2834                -1.4672 3505.1
EM variationnel, Kmax = 6 (4 acti

,periode,observations,moyenne_pct,ecart_type_pct,asymetrie,kurtosis_excedentaire,minimum_pct,maximum_pct
0,apprentissage 2016-2021,1318,0.060,1.1960,-1.154,22.720,-12.77,8.97
1,évaluation 2022-2026,1193,0.041,1.0786,0.039,6.487,-6.16,9.09


,modele,composantes,loglik_moy_apprentissage,loglik_moy_evaluation,BIC
0,"Gaussienne unique (EM, K = 1)",1,-1.5976,-1.5043,4225.5
1,"EM classique, K choisi par BIC (K = 4)",4,-1.2876,-1.4620,3473.1
2,"EM classique, K = 6 imposé",6,-1.2834,-1.4672,3505.1
3,"EM variationnel, Kmax = 6 (4 actives), prédict...",4,-1.2904,-1.4599,NaN


In [18]:
display(pd.read_csv(M.SORTIE / "cas_pratique_var.csv"))
M.FIGURES["cas_var"].show()

,niveau,modele,VaR_pct,ES_pct,IC95_VaR,exceptions,exceptions_attendues,taux_observe,Kupiec_LR,Kupiec_p
0,0.05,Gaussienne unique,-1.907,-2.406,NaN,41.0,59.7,0.0344,6.86,0.0088
1,0.05,Mélange EM (K = 4),-1.674,-3.063,NaN,57.0,59.7,0.0478,0.13,0.7229
2,0.05,Mélange variationnel (prédictive),-1.685,-3.124,[-1.948 ; -1.454],57.0,59.7,0.0478,0.13,0.7229
3,0.05,"Bootstrap EM (K = 4, B = 1000)",-1.701,NaN,[-2.085 ; -1.369],NaN,NaN,NaN,NaN,NaN
4,0.01,Gaussienne unique,-2.721,-3.126,NaN,20.0,11.9,0.0168,4.58,0.0323
5,0.01,Mélange EM (K = 4),-3.665,-5.665,NaN,6.0,11.9,0.0050,3.64,0.0563
6,0.01,Mélange variationnel (prédictive),-3.717,-5.862,[-4.475 ; -3.217],5.0,11.9,0.0042,5.20,0.0225
7,0.01,"Bootstrap EM (K = 4, B = 1000)",-3.725,NaN,[-4.723 ; -2.984],NaN,NaN,NaN,NaN,NaN


In [19]:
display(pd.read_csv(M.SORTIE / "cas_pratique_hmm.csv"))
M.FIGURES["cas_regimes"].show()

,etat,moyenne_pct_jour,volatilite_annualisee_pct,probabilite_stationnaire,duree_moyenne_jours
0,calme,0.1233,8.74,0.761,45.0
1,agité,-0.1423,35.42,0.239,14.1


## 16bis. Scénarios synthétiques S1 à S10 et comparaison avec EM-BIC et EM-ICL
Chaque scénario compte au moins 30 réplications et compare EM-K, EM-Kmax, EM-BIC, EM-ICL, MAP-EM et le VB avec le **même critère d'arrêt**. Durée complète : plusieurs heures (un processus par scénario conseillé, voir `A_REGENERER.md`). `RAPIDE` lance une version réduite.

In [20]:
import subprocess, sys
def lancer(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout[-2000:] or r.stderr[-2000:])
lancer('scenarios.py', *(['--rapide'] if RAPIDE else []), 'S1', 'S2', 'S3')   # priorité absolue ; les autres : S4 à S10
lancer('scenarios.py', '--regrille')

== S1 ==
== S2 ==
== S3 ==




In [21]:
display(pd.read_csv(M.SORTIE / 'grille_scenarios.csv')[['id', 'decision', 'estimation', 'intervalle', 'p_holm']])

,id,decision,estimation,intervalle,p_holm
0,S1[1],équivalent,0.010195,[0.002 ; 0.020],6.626750e-05
1,S1[2],A > B,0.166867,[0.126 ; 0.209],3.410657e-05
2,S1[3],A > B,0.279663,[0.239 ; 0.321],2.254717e-05
3,S1[4],équivalent,-0.000938,[-0.011 ; 0.005],2.476019e-10
4,S1[5],équivalent,0.002524,[0.000 ; 0.005],7.749034e-132
5,S2[80/15/5],équivalent,-0.033333,[-0.267 ; 0.233],5.605703e-04
6,S3[Kvrai=4],A > B,0.105179,[0.086 ; 0.125],2.353421e-05
7,S3[Kvrai=5],A > B,0.096957,[0.068 ; 0.129],3.428549e-05
8,S3[Kvrai=6],A > B,0.139188,[0.124 ; 0.155],2.254717e-05
9,S4[Student nu=3],A > B,0.063126,[0.044 ; 0.082],6.626750e-05


## 16ter. Cas principal : régimes climatiques du Cameroun
Données WorldClim 2.1 (téléchargement par `python cas_pratique_climat.py --preparer`, redistribution interdite) et Köppen–Geiger (Beck et al., 2018). Le zonage agro-écologique officiel n'étant pas disponible, la référence est Köppen–Geiger seul. Hypothèses H1 à H8 fixées avant l'analyse (tableau du chapitre 3).

In [ ]:
K = ['--koppen', 'data/climat/koppen/Beck_KG_V1_present_0p0083.tif', '--frontiere', 'data/climat/frontiere/cameroun.shp']
if (M.DOSSIER / 'data' / 'climat' / 'wc2.1_2.5m_elev.tif').exists():
    lancer('cas_pratique_climat.py', *K, *(['--rapide'] if RAPIDE else []))
else:
    print('Données absentes : lancer python cas_pratique_climat.py --preparer')

In [ ]:
g = M.SORTIE / 'climat' / 'grille_hypotheses_climat.csv'
if g.exists(): display(pd.read_csv(g)[['id', 'decision', 'estimation', 'intervalle', 'p_holm']])

## 16quater. S&P 500 : compléments (Student-t, GARCH-t, simulation historique, Christoffersen, ES, filtrage forward)

In [ ]:
lancer('cas_pratique_sp500_complement.py', *(['--rapide'] if RAPIDE else []))
display(pd.read_csv(M.SORTIE / 'sp500_backtests.csv'))

## 16quinquies. pLSA contre LDA sur un corpus réel (20 Newsgroups)

In [ ]:
lancer('texte_plsa_lda.py', *(['--rapide'] if RAPIDE else []))
display(pd.read_csv(M.SORTIE / 'texte_plsa_lda.csv'))

## 16sexies. Production des tableaux du mémoire
`tex_tables.py` écrit `tables/*.tex` à partir de `resultats/` : aucun nombre n'est saisi à la main.

In [ ]:
lancer('tex_tables.py')

## 17. Export des figures
Copie des figures PNG vers `figures/` (noms `fig01.png` à `fig12.png`) et des tableaux vers `resultats/`.

In [40]:
M.exporter_figures()
print(sorted(p.name for p in (M.DOSSIER / "figures").glob("fig*.png")))

['fig01.png', 'fig02.png', 'fig03.png', 'fig04.png', 'fig05.png', 'fig06.png', 'fig07.png', 'fig08.png', 'fig09.png', 'fig10.png', 'fig11.png', 'fig12.png', 'fig13.png', 'fig14.png', 'fig15.png', 'fig16.png']
